# k-Nearest Neighbors (KNN) – verständlich & mit PoC

### Was ist KNN?
- **Idee:** Für einen Datenpunkt die *k* ähnlichsten Nachbarn finden (z. B. per euklidischer Distanz).  
- **Klassifikation:** Klasse = Mehrheitsvotum der Nachbarn.  
- **Regression / Imputation:** Wert = Durchschnitt/Median (optional distanzgewichtet).  
- **Merke:** Einfach zu verstehen, braucht aber **skalierte Features** und eine sinnvolle Wahl von $ k $.


<p align="center">
  <img src="https://miro.medium.com/v2/resize:fit:786/format:webp/0*nBQ9HKuxTaMtR0Bi.png" width="500">
</p>

*Abbildung: Ein neuer Punkt wird der Klasse zugeordnet, die unter seinen k nächsten Nachbarn überwiegt.*

### Wahl von k
- **Klein** (z. B. $k=1$): sehr empfindlich gegenüber Ausreißern, hohe Varianz.
- **Groß**: glattere Entscheidungen, aber weniger Details.
- **Tipp**: Bei binärer Klassifikation ungerade k-Werte wählen, um Gleichstand zu vermeiden.


<p align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*OyYyr9qY-w8RkaRh2TKo0w.png" width="500">
</p>

*Abbildung 2: Einfluss von k – kleine Werte führen zu hoher Varianz (überanpasst), große Werte zu hohem Bias (unteranpasst).*


### Ablauf im PoC (Illustration)
<p align="center">
  <img src="https://insightimi.wordpress.com/wp-content/uploads/2020/03/knn-start.png" width="500">
</p>
 
*Abbildung: Visualisierung des KNN-Ablaufs – Datenausgang → Distanzmessung → Nachbarn bestimmen → Klassenvotum.*


### 1. Distanzmetriken
### Welche Distanzmetrik wann?
| Metrik        | Geeignet für                       | Bemerkung |
|---------------|------------------------------------|-----------|
| Euklidisch    | kontinuierliche Daten              | Standardfall, misst direkte Luftlinie |
| Manhattan     | kontinuierliche Daten mit Ausreißern | Robust gegen einzelne große Unterschiede |
| Minkowski     | allgemeine Form von Euklidisch/Manhattan | p=1 → Manhattan, p=2 → Euklidisch |
| Hamming       | kategoriale/Binärdaten              | Zählt Positionen mit Unterschieden |


### Euklidische Distanz
$$
d(p,q) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}
$$

### Manhattan-Distanz (\( p=1 \))
$$
d(x,y) = \sum_{i=1}^m |x_i - y_i|
$$

### Minkowski-Distanz
$$
d(x,y) = \left( \sum_{i=1}^n |x_i - y_i|^p \right)^{\frac{1}{p}}
$$

### Hamming-Distanz
$$
D_H = \sum_{i=1}^k |x_i - y_i|
$$


### 2. Erklärungen zu Symbolen
- $N_k(x)$ = Menge der $k$ nächsten Nachbarn von $( x )$.  
- $ p $ = Parameter der Minkowski-Distanz ($p=1$ → Manhattan, $p=2$ → Euklidisch).  
- $ \epsilon$ = kleiner Wert, um Division durch Null zu vermeiden.  
- $ w_j$ = Gewicht eines Nachbarn $( j )$, oft $( w_j = \frac{1}{d(x, x_j) + \epsilon})$.  
- $( m, n, k)$ = Anzahl Dimensionen oder Nachbarn.  



### 3. Klassifikation (Mehrheit)
$$
\hat{y} = \operatorname{mode}\big( y_j \;|\; j \in N_k(x) \big)
$$


### 4. Regression (distanzgewichtet)
$$
\hat{y}k = \frac{\sum_{j \in N_k(x)} w_j \cdot y_j}{\sum_{j \in N_k(x)} w_j}
$$



### 5. Python-Beispiel mit scikit-learn
```python
from sklearn.neighbors import KNeighborsClassifier

# Modell erstellen
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)

# Training
knn.fit(X_train, y_train)

# Vorhersage
y_pred = knn.predict(X_test)
```

### Wann ist KNN sinnvoll?
✅ Kleine bis mittelgroße Daten, **ähnliche Punkte → ähnliche Werte**  
✅ Fehlende Werte über Nachbarn schätzen (Imputation)  
⚠️ Features **skalieren** (sonst dominiert ein Maßstab)  
⚠️ Bei sehr großen Daten teuer in der Berechnung  


### Skalierung der Features
- KNN basiert auf Distanzen → große Wertebereiche dominieren sonst.
- **Lösung**: Vorher StandardScaler oder MinMaxScaler verwenden.


In [14]:
# Fehlende Werte: schnelle Pandas-Optionen

#Wir importieren numpy (für Arrays) und den KNN-Klassifikator aus scikit-learn.
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# 1D-Feature: nur "Süße"
X_train = np.array([[2],[3],[4],   # Banane
                    [7],[8],[9]])  # Apfel
y_train = np.array(['Banane','Banane','Banane',
                    'Apfel','Apfel','Apfel'])

# Wir erstellen einen KNN-Klassifikator mit k=3 (schaut sich also immer 3 Nachbarn an).
# Danach wird das Modell mit unseren Daten trainiert.
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

X_test = np.array([[3],[8],[5]])
print(knn.predict(X_test))
# -> ['Banane' 'Apfel' 'Banane']
# Wir testen drei neue Früchte:

# [3] → liegt nah bei den Bananen → Banane
# [8] → liegt nah bei den Äpfeln → Apfel
# [5] → liegt genau in der Mitte → schaut auf die 3 nächsten Nachbarn → 2 Bananen, 1 Apfel → Banane


['Banane' 'Apfel' 'Banane']


In [2]:
# === PoC: KNN‑Imputation von fehlenden Werten (kompakt & kommentiert) ===
# Falls nötig: !pip install scikit-learn

import pandas as pd
import numpy as np

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Beispiel-Daten mit Lücken
df = pd.DataFrame({
    "gruppe": ["A","A","A","B","B","B"],
    "alter":  [23, 25, np.nan, 35, np.nan, 40],
    "eink":   [3200, np.nan, 2900, np.nan, 4100, 4500],   # Einkommen
    "score":  [0.6, 0.72, np.nan, 0.55, 0.8, np.nan]
})
print("Original:\n", df, "\n")

# 1) Numerische Spalten auswählen (KNNImputer arbeitet auf Zahlen)
num_cols = ["alter", "eink", "score"]
num = df[num_cols].copy()

# 2) Skalieren (wichtig für sinnvolle Distanzen)
scaler = StandardScaler()
num_scaled = scaler.fit_transform(num)

# 3) KNN-Imputation
imputer = KNNImputer(n_neighbors=3, weights="distance")  # k=3 ist ein solider Start
num_imputed_scaled = imputer.fit_transform(num_scaled)

# 4) Zurückskalieren + zurück in den DataFrame
df_imputed = df.copy()
df_imputed[num_cols] = scaler.inverse_transform(num_imputed_scaled)

# Optional: bestimmte Spalten runden (z. B. Alter = ganze Jahre)
df_imputed["alter"] = df_imputed["alter"].round().astype(int)

print("Nach KNN-Imputation:\n", df_imputed, "\n")


Original:
   gruppe  alter    eink  score
0      A   23.0  3200.0   0.60
1      A   25.0     NaN   0.72
2      A    NaN  2900.0    NaN
3      B   35.0     NaN   0.55
4      B    NaN  4100.0   0.80
5      B   40.0  4500.0    NaN 

Nach KNN-Imputation:
   gruppe  alter         eink     score
0      A     23  3200.000000  0.600000
1      A     25  3806.714785  0.720000
2      A     26  2900.000000  0.640000
3      B     35  4041.278365  0.550000
4      B     32  4100.000000  0.800000
5      B     40  4500.000000  0.688932 



### Euklidische Distanz  
**Definition:**  
Misst die „Luftlinien“-Distanz (geradlinig) zwischen zwei Punkten.  
Wird oft in Geometrie und KNN verwendet, wenn der direkte Abstand wichtig ist.

**Formel:**  
$$
d(x,y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}
$$




### Manhattan-Distanz
$$
d(x,y) = \sum_{i=1}^{n} |x_i - y_i|
$$



### Minkowski-Distanz

**Definition:**  
Die Minkowski-Distanz ist eine **verallgemeinerte Form** der Euklidischen und der Manhattan-Distanz.  
Sie hat einen **Parameter \(p\)**, mit dem man zwischen verschiedenen Distanzarten wechseln kann:

- $p = 1$ → **Manhattan-Distanz**
- $p = 2$ → **Euklidische Distanz**
- andere $(p)$-Werte → andere Distanzmetriken

**Formel:**  
$$
d(x,y) = \left( \sum_{i=1}^n |x_i - y_i|^p \right)^{\frac{1}{p}}
$$


**Bedeutung der Symbole:**
- $(x_i, y_i)$ = Koordinaten der Punkte $(x)$ und $(y)$ in Dimension $(i)$
- $n$ = Anzahl der Dimensionen
- $p$ = Parameter, der die Art der Distanz bestimmt
- $|x_i - y_i|$ = Abstand in einer Dimension



**Beispiel:**

Wir vergleichen zwei Punkte:

$$
x = (1, 2), \quad y = (4, 6)
$$

**Euklidische Distanz p=2:**
$$
d = \sqrt{ (1-4)^2 + (2-6)^2 }
   = \sqrt{ (-3)^2 + (-4)^2 }
   = \sqrt{ 9 + 16 }
   = \sqrt{25} = 5
$$



**Manhattan-Distanz p=1:**
$$
d = |1-4| + |2-6|
   = 3 + 4
   = 7
$$



**Minkowski mit p=3:**
$$
d = \left( |1-4|^3 + |2-6|^3 \right)^{\frac{1}{3}}
   = \left( 27 + 64 \right)^{\frac{1}{3}}
   = (91)^{\frac{1}{3}} \approx 4.497
$$





 **Merke:**  
- **Kleinere $p-Werte$** → Betonung der **einzelnen Koordinatenunterschiede** (ähnlich Manhattan)  
- **Größere $p-Werte$** → Betonung der **größten Unterschiede** (ähnlich Euklidisch oder darüber hinaus)  
- Für $p \to \infty$ misst man im Wesentlichen den **maximalen Einzelunterschied** (Chebyshev-Distanz).
- 

### Chebyshev-Distanz

**Definition**  
Misst den größten Unterschied in einer einzelnen Koordinate zwischen zwei Punkten.  
Auch bekannt als **$L_\infty-Norm$**.

**Formel**  
$$
d(x, y) = \max_{i} |x_i - y_i|
$$

**Eigenschaft**  
- Entspricht der Minkowski-Distanz mit $$p \to \infty$$.
- Betont nur den **größten Einzelunterschied** und ignoriert die anderen.
- Oft verwendet bei Schachbrett-Distanzen (Königsbewegung).

**Beispiel**  
$$
x = (1, 2), \quad y = (4, 6)$$

$$d = \max(|1-4|, |2-6|) = \max(3, 4) = 4
$$




### Hamming-Distanz

**Definition:**  
Die Hamming-Distanz zählt, an wie vielen Positionen sich zwei Vektoren (z. B. Strings oder Binärwerte) unterscheiden.  
Sie wird oft in der Fehlererkennung, bei Datenübertragungen oder DNA-Analysen verwendet.

**Formel:**  
$$
D_H(x,y) = \sum_{i=1}^n \delta(x_i, y_i),
\quad
\delta(a,b) =
\begin{cases}
0 & \text{wenn } a=b \\
1 & \text{wenn } a \neq b
\end{cases}
$$

**Beispiel:**

$$ x = \texttt{1011101} $$ 
$$y = \texttt{1001001}$$
 

Vergleich **Stelle für Stelle**:  

| Position \(i\) | \(x_i\) | \(y_i\) | Gleich? | Unterschied (0/1) |
|----------------|--------|--------|---------|--------------------|
| 1              | 1      | 1      | ✅ Ja   | 0                  |
| 2              | 0      | 0      | ✅ Ja   | 0                  |
| 3              | 1      | 0      | ❌ Nein | 1                  |
| 4              | 1      | 1      | ✅ Ja   | 0                  |
| 5              | 1      | 0      | ❌ Nein | 1                  |
| 6              | 0      | 0      | ✅ Ja   | 0                  |
| 7              | 1      | 1      | ✅ Ja   | 0                  |

**Summe der Unterschiede:**  
$$
D_H = 0 + 0 + 1 + 0 + 1 + 0 + 0 = 2
$$

➡ **Ergebnis:** Die Hamming-Distanz beträgt **2** – die beiden Strings unterscheiden sich an **zwei Positionen**.


###  Merke:
- **Euklidisch:** direkte Luftlinie  
- **Manhattan:** Achsenwege (Stadtblöcke)  
- **Minkowski:** verallgemeinert beide  
- **Hamming:** zählt Unterschiede (keine geometrische Distanz)


### Vor- und Nachteile von KNN

✅ **Vorteile**:
- Einfach zu verstehen und zu implementieren.
- Keine explizite Trainingsphase.

❌ **Nachteile**:
- Langsam bei großen Datensätzen (muss alle Distanzen berechnen).
- Speicherintensiv (braucht alle Trainingsdaten).
- Empfindlich gegenüber irrelevanten oder unskalierten Features.
